# RAFNet origin-head training (Colab / Kaggle GPU or CPU)
1. Upload `data/embed_cache/*.pt` shards + `manifest` info baked in each row.
2. Run all: trains origin MLP (5 classes), generator-family classifier with open-set
   energy threshold, IVR head, logistic stacking with RAFNet scores, temperature scaling.
3. NEVER touches the test split for tuning. Download `origin_classifier.pt` → repo `models/`.

In [ ]:
!pip install -q torch scikit-learn matplotlib
import torch, torch.nn as nn, numpy as np, glob, random
from pathlib import Path
from collections import Counter
SEED = 7; random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

In [ ]:
CACHE = Path('/content/embed_cache')  # <-- adjust: uploaded shards dir
rows = []
for f in sorted(glob.glob(str(CACHE / '*.pt'))):
    rows += torch.load(f, map_location='cpu', weights_only=False)
print(len(rows), Counter(r['split'] for r in rows))
print(Counter(r['label_origin'] for r in rows))

In [ ]:
ORIGIN = ['HUMAN_LIVE','HUMAN_RECORDED_REPLAY','AI_TTS','AI_VOICE_CLONE_OR_CONVERSION','UNCERTAIN']
def pack(split, key='label_origin', classes=None):
    sel = [r for r in rows if r['split'] == split]
    X = np.stack([r['x'] for r in sel])
    y = np.array([classes.index(r[key]) for r in sel])
    return X, y
Xtr, ytr = pack('train'); Xva, yva = pack('val'); Xte, yte = pack('test')
print(Xtr.shape, Xva.shape, Xte.shape)

In [ ]:
class Head(nn.Module):
    def __init__(self, d_in, n_cls, h=256):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d_in, h), nn.LayerNorm(h), nn.GELU(), nn.Dropout(0.2), nn.Linear(h, n_cls))
    def forward(self, x):
        return self.net(x)

def train_head(Xtr, ytr, Xva, yva, n_cls, epochs=60, lr=3e-4, patience=8):
    from collections import Counter
    cnt = Counter(ytr.tolist()); w = torch.tensor([len(ytr)/cnt[c] for c in range(n_cls)])
    dev = 'cuda' if torch.cuda.is_available() else 'cpu'
    m = Head(Xtr.shape[1], n_cls).to(dev); opt = torch.optim.AdamW(m.parameters(), lr=lr)
    ce = nn.CrossEntropyLoss(weight=w.to(dev))
    Xt, Xv = torch.from_numpy(Xtr).float(), torch.from_numpy(Xva).float()
    yt, yv = torch.from_numpy(ytr).long(), torch.from_numpy(yva).long()
    best, wait, best_state = -1, 0, None
    for ep in range(epochs):
        m.train()
        for i in range(0, len(Xt), 256):
            opt.zero_grad(); loss = ce(m(Xt[i:i+256].to(dev)), yt[i:i+256].to(dev)); loss.backward(); opt.step()
        m.eval()
        from sklearn.metrics import f1_score
        with torch.inference_mode():
            f1 = f1_score(yv.numpy(), m(Xv.to(dev)).argmax(1).cpu().numpy(), average='macro')
        print(f'ep{ep} val macro-F1 {f1:.4f}')
        if f1 > best + 1e-4: best, wait, best_state = f1, 0, {k: v.cpu() for k, v in m.state_dict().items()}
        else:
            wait += 1
            if wait >= patience: break
    m.load_state_dict(best_state); return m, best

origin_head, f1 = train_head(Xtr, ytr, Xva, yva, len(ORIGIN))
print('origin val macro-F1:', f1)

In [ ]:
# Generator-family head + open-set energy threshold (fit on VAL only).
FAMS = sorted({r['generator_family'] for r in rows})
XtrF, ytrF = pack('train', 'generator_family', FAMS)
XvaF, yvaF = pack('val', 'generator_family', FAMS)
fam_head, fam_f1 = train_head(XtrF, ytrF, XvaF, yvaF, len(FAMS))
print('family val macro-F1:', fam_f1, FAMS)
fam_head.eval()
with torch.inference_mode():
    dev = next(fam_head.parameters()).device
    logits = fam_head(torch.from_numpy(XvaF).float().to(dev))
    energy = -torch.logsumexp(logits, dim=1).cpu().numpy()
ENERGY_TAU = float(np.quantile(energy, 0.95))  # 5% of VAL treated unknown; report it
print('energy tau (95th pct of val):', ENERGY_TAU)

In [ ]:
# Temperature scaling on VAL + chosen UNCERTAIN band from VAL coverage curve.
from sklearn.metrics import confusion_matrix, f1_score
origin_head.eval()
with torch.inference_mode():
    dev = next(origin_head.parameters()).device
    zva = origin_head(torch.from_numpy(Xva).float().to(dev))
T = torch.nn.Parameter(torch.ones(1))
opt = torch.optim.LBFGS([T], lr=0.1, max_iter=50)
ce = nn.CrossEntropyLoss()
def ev():
    opt.zero_grad(); loss = ce(zva / T.clamp(min=1e-3), torch.from_numpy(yva).long().to(dev)); loss.backward(); return loss
opt.step(ev)
print('temperature:', float(T))
torch.save({'origin': origin_head.state_dict(), 'family': fam_head.state_dict(),
            'classes_origin': ORIGIN, 'families': FAMS, 'temperature': float(T),
            'energy_tau': ENERGY_TAU, 'val_macro_f1': f1, 'seed': SEED},
           '/content/origin_classifier.pt')
print('saved. NOW evaluate on test ONCE (see train_plan + eval script), then write the model card.')